# Adem coboundaries: the even prime

This notebook follows the paper [*A cochain level proof of Adem relations in the mod 2 Steenrod algebra*](../papers/AdemRelations.pdf) and uses `comch` to construct its Adem coboundaries in the Barratt-Eccles and surjection operads.

## Introduction

For mod-$2$ cohomology, the Steenrod squares satisfy the Adem relations

$$
Sq^a Sq^b = \sum_i \binom{b-1-i}{a-2i} Sq^{a+b-i} Sq^i,
\qquad a < 2b.
$$

The paper proves these relations at the cochain level. Instead of only showing that a certain sum of Steenrod operations vanishes in cohomology, it constructs explicit cochains whose coboundaries are the relevant Adem sums.

In `comch`, the pair of integers $(q,p)$ with $q > p \geq 0$ indexes the universal Barratt-Eccles homotopy used in the paper. After table reduction to the surjection operad, this homotopy can be evaluated on simplicial or cubical chains. The result is a chain representative of an Adem coboundary for any homogeneous mod-$2$ cocycle $\alpha$.

## The Barratt-Eccles Adem homotopy

Let

$$
\widetilde{x}_i = (e,(12),e,\ldots,(12)^i) \in \mathcal E(2).
$$

The Adem construction uses two homotopies. The first is the homotopy $J_\Psi$ between the inclusion of $EV_4$ and the map $\Psi = EZ \circ \Phi \circ AW$ built from the Alexander-Whitney and Eilenberg-Zilber maps. The second is the homotopy $J_{(23)}$ associated to the inner automorphism by the transposition $(23) \in \Sigma_4$.

The method `BarrattEccles.adem_homotopy(q, p)` returns the normalized Barratt-Eccles element

$$
J_\Psi(\widetilde{x}_q \times \widetilde{x}_p)
+ J_\Psi(\widetilde{x}_p \times \widetilde{x}_q)
+ J_{(23)}(\widetilde{x}_q \times \widetilde{x}_p)
\in \mathcal E(4).
$$

This is the universal chain whose boundary gives the corresponding Adem relator.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start):
    for path in (start, *start.parents):
        if (path / 'comch').is_dir() and (path / 'notebooks').is_dir():
            return path
    return start


repo_root = find_repo_root(Path.cwd()).resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

loaded_comch = sys.modules.get('comch')
if loaded_comch is not None:
    module_path = Path(getattr(loaded_comch, '__file__', '')).resolve()
    if not str(module_path).startswith(str(repo_root)):
        for name in list(sys.modules):
            if name == 'comch' or name.startswith('comch.'):
                del sys.modules[name]

from comch import BarrattEccles


def preview(element, max_terms=8):
    """Return a readable preview of a possibly long chain."""
    if len(element) <= max_terms:
        return str(element)
    head = dict(list(element.items())[:max_terms])
    shown = element.create(head)
    return f'{shown} + ... ({len(element) - max_terms} more terms)'

In [2]:
for q, p in [(1, 0), (2, 1)]:
    homotopy = BarrattEccles.adem_homotopy(q, p)
    print(f'A_{{{q},{p}}} in E(4): degree {homotopy.degree}, {len(homotopy)} terms')
    print(preview(homotopy))
    print()

A_{1,0} in E(4): degree 2, 3 terms
((1,2,3,4),(1,2,4,3),(2,1,4,3)) + ((1,2,3,4),(1,3,2,4),(3,1,4,2)) + ((1,2,3,4),(3,4,1,2),(3,1,4,2))

A_{2,1} in E(4): degree 4, 33 terms
((1,2,3,4),(1,2,4,3),(2,1,4,3),(4,3,2,1),(2,1,4,3)) + ((1,2,3,4),(1,2,4,3),(4,3,1,2),(4,3,2,1),(2,1,4,3)) + ((1,2,3,4),(1,2,4,3),(4,3,1,2),(1,2,4,3),(2,1,4,3)) + ((1,2,3,4),(3,4,1,2),(4,3,1,2),(4,3,2,1),(2,1,4,3)) + ((1,2,3,4),(3,4,1,2),(4,3,1,2),(1,2,4,3),(2,1,4,3)) + ((1,2,3,4),(3,4,1,2),(1,2,3,4),(1,2,4,3),(2,1,4,3)) + ((1,2,3,4),(2,1,4,3),(1,2,4,3),(1,2,3,4),(3,4,1,2)) + ((1,2,3,4),(2,1,4,3),(1,2,4,3),(4,3,1,2),(3,4,1,2)) + ... (25 more terms)



## Table reduction

The table reduction morphism $TR \colon \mathcal E \to \mathcal X$ sends Barratt-Eccles elements to surjection elements. Since the Adem homotopies here are defined over $\mathbb F_2$, the Berger-Fresse and McClure-Smith conventions have the same coefficients.

The method `Surjection.adem_homotopy(q, p)` is the table reduction of `BarrattEccles.adem_homotopy(q, p)`.

In [ ]:
from comch import Surjection


for q, p in [(1, 0), (2, 1)]:
    surjection = Surjection.adem_homotopy(q, p)
    print(f'TR(A_{{{q},{p}}}) in X(4): degree {surjection.degree}, {len(surjection)} terms')
    print(preview(surjection))
    print()

## Action on simplices and cubes

Simplicial and cubical cochains are algebras over the surjection operad. For a mod-$2$ cocycle $\alpha$, the Adem coboundary associated to $(q,p)$ is determined by the action of the surjection element on chains:

$$
\eta_{q,p}(\alpha)(c) =
(\alpha \otimes \alpha \otimes \alpha \otimes \alpha) a_{q,p}(c).
$$

The method `Steenrod.adem_coboundaries(q, p, n)` evaluates this chain map on the top $n$-simplex. Passing `shape='cube'` evaluates it on the top $n$-cube. Since $\alpha$ is homogeneous, the output is filtered to summands where all four tensor factors have the same dimension.

In [ ]:
from comch import Steenrod


examples = [
    (1, 0, 3, 'simplex'),
    (2, 1, 4, 'simplex'),
    (1, 0, 1, 'cube'),
]

for q, p, n, shape in examples:
    answer = Steenrod.adem_coboundaries(q, p, n, shape=shape)
    print(f'A_{{{q},{p}}} on the top {n}-{shape}: {len(answer)} terms')
    print(preview(answer))
    print()

## A larger cubical example

The cubical action is available through the same method, but nontrivial Adem examples can grow much faster than the Cartan examples. The next cell is disabled by default. Set `RUN_LARGE_CUBE_EXAMPLE = True` to compute a nonzero cubical example.

In [ ]:
RUN_LARGE_CUBE_EXAMPLE = False

if RUN_LARGE_CUBE_EXAMPLE:
    cube_example = Steenrod.adem_coboundaries(2, 1, 4, shape='cube')
    print(f'A_{{2,1}} on the top 4-cube: {len(cube_example)} terms')
    print(preview(cube_example, max_terms=6))
else:
    print('Set RUN_LARGE_CUBE_EXAMPLE = True to compute the 4-cube example.')

## References

[BMM] G. Brumfiel, A. Medina-Mardones, and J. Morgan. "A cochain level proof of Adem relations in the mod 2 Steenrod algebra."

[BF] C. Berger and B. Fresse. "Combinatorial operad actions on cochains." *Mathematical Proceedings of the Cambridge Philosophical Society* 137 (2004), no. 1, 135-174. [DOI](https://doi.org/10.1017/S0305004103007138)